# Submit an LLM Job via the i2b2 ETL REST API

Submits a `job_type='llm'` job through the Flask API (instead of a direct SQL insert),
polls until the jobWatcher completes it, and shows the extraction output.

**Prerequisites** (from `deployment/pg`):

```
docker compose up -d i2b2-pg i2b2-etl i2b2-ml ollama i2b2-wildfly
```

- `i2b2-pg` - database
- `i2b2-etl` - this REST API (host port **5001** -> container 5000)
- `i2b2-ml` - jobWatcher that picks up and runs the job
- `ollama` - local LLM endpoint (`qwen3:0.6b`)
- `i2b2-wildfly` - PM service, used by the API to validate credentials

Test data (patient 99001, note concept `/i2b2/Notes/ClinicalNotes/`, output concept
`/i2b2/LLM/NLP/Extract/`) must already be seeded - see `docs/HOW_TO_RUN_LLM_ENGINE.md`.

In [ ]:
import requests
import json
import base64
import time

# --- Config ---
BASE_URL = "http://localhost:5001"   # from the host; use http://i2b2-etl:5000 inside the docker network (e.g. i2b2-jupyter)
PROJECT  = "Demo"                    # i2b2 project name (X-Project-Name header)
USERNAME = "demo"                    # i2b2 username
PASSWORD = "Etl@2021"                # I2B2_ROOT_PASSWORD from deployment/pg/.env

## 1. Auth headers

The API uses HTTP Basic auth with username `Project\Username`.
The password is either an active i2b2 session key **or** the plain i2b2 password -
if it isn't a known session, the API asks the PM service (wildfly) to create one.

In [ ]:
def make_headers(project, username, password):
    credentials = f"{project}\\{username}:{password}"
    encoded = base64.b64encode(credentials.encode("ascii")).decode("ascii")
    return {
        "Authorization": f"Basic {encoded}",
        "X-Project-Name": project,
        "Content-Type": "application/json",
    }

HEADERS = make_headers(PROJECT, USERNAME, PASSWORD)

def show(response):
    print(f"Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text)

## 2. Submit the LLM job - `POST /etl/job`

| input field | meaning |
|---|---|
| `path` | concept path of the output concept (must exist in `concept_dimension`) |
| `note_concept_path` | concept path prefix for finding clinical notes in `observation_fact` |
| `search_terms` | terms to detect in each note |
| `output_concept_code` | base concept code for the result facts (child codes `CODE:term` are auto-created) |
| `vllm_url` | OpenAI-compatible endpoint, **without** `/v1` (ollama or vLLM) |
| `model` | model name as served by the endpoint |

The jobWatcher polls the job table every 10 s, so pickup is nearly immediate.

In [ ]:
job = {
    "jobType": "llm",
    "input": {
        "path": "/i2b2/LLM/NLP/Extract/",
        "note_concept_path": "/i2b2/Notes/ClinicalNotes/",
        "search_terms": ["metformin", "HbA1c", "insulin"],
        "output_concept_code": "LLM:NLP:FOUND",
        # local ollama (docker-network hostname; the jobWatcher resolves it, not this notebook)
        "vllm_url": "http://ollama:11434",
        "model": "qwen3:0.6b",
    },
}

r = requests.post(f"{BASE_URL}/etl/job", headers=HEADERS, json=job)
show(r)

## 3. Find the job id

The POST response is just a message, so grab the newest job from `GET /etl/job`
(jobs are returned newest-first).

In [ ]:
r = requests.get(f"{BASE_URL}/etl/job", headers=HEADERS)
r.raise_for_status()
latest = r.json()[0]
JOB_ID = latest["id"]
print(f"Job id: {JOB_ID}  status: {latest['status']}")

## 4. Poll until the job finishes

`PENDING -> PROCESSING -> COMPLETED` (or `ERROR`). With the local CPU model this
takes ~20-60 s for one note and three terms.

In [ ]:
for _ in range(60):
    r = requests.get(f"{BASE_URL}/etl/job", headers=HEADERS, params={"jobId": JOB_ID})
    r.raise_for_status()
    job_row = r.json()[0]
    print(f"status: {job_row['status']}")
    if job_row["status"] in ("COMPLETED", "ERROR"):
        break
    time.sleep(10)

print()
print("Output:")
print(json.dumps(job_row["output"], indent=2))

## 5. Interpret the result

The output maps each patient's MRN to a `term -> found` boolean, e.g.

```json
{"99001": {"metformin": true, "HbA1c": true, "insulin": false}}
```

The same values are written to `observation_fact` as facts with concept codes
`LLM:NLP:FOUND:<term>` and `tval_char` `1`/`0` (the patient appears under a
de-identified `patient_num`). Rerunning the job replaces those facts.

To verify in the database:

```
docker exec i2b2-pg psql -U postgres -d i2b2 -c "SELECT patient_num, concept_cd, tval_char FROM i2b2demodata.observation_fact WHERE concept_cd LIKE 'LLM:NLP:FOUND%'"
```

## 6. (Optional) Re-run a job

`POST /etl/update_job` flips a job's status if it currently matches `pre` -
e.g. reset a completed job back to `PENDING` so the watcher runs it again,
or recover a job stuck in `PROCESSING` after a watcher crash.

In [ ]:
# r = requests.post(f"{BASE_URL}/etl/update_job", headers=HEADERS,
#                   json={"jobId": JOB_ID, "pre": "COMPLETED", "post": "PENDING"})
# show(r)